In [20]:
import pprint
# Load environment variables and verify the project setup.
import sys
from pathlib import Path

# Find the repo root (the folder containing env_checker.py) and make it importable.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "env_checker.py").exists())
sys.path.insert(0, str(ROOT))

# Load .env into the environment for this session.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ModuleNotFoundError:
    print("python-dotenv not installed yet — run: uv add python-dotenv")

# Verify .env variables and required packages.
from env_checker import run_checks
run_checks()

Environment variables (from .env.example)
  ✓ OPENAI_API_KEY  — set
  ✓ ANTHROPIC_API_KEY  — set
  ✓ LANGSMITH_TRACING  — set
  ✓ LANGSMITH_ENDPOINT  — set
  ✓ LANGSMITH_API_KEY  — set
  ✓ LANGSMITH_PROJECT  — set
  ✓ CHROMA_PERSIST_DIR  — set

Required packages (from pyproject.toml)
  ✓ beautifulsoup4  — installed (4.14.3)
  ✓ chromadb  — installed (1.5.9)
  ✓ langchain  — installed (1.3.2)
  ✓ langchain-chroma  — installed (1.1.0)
  ✓ langchain-community  — installed (0.4.2)
  ✓ langchain-openai  — installed (1.2.2)
  ✓ lxml  — installed (6.1.1)
  ✓ onnxruntime  — installed (1.19.2)
  ✓ pypdf  — installed (6.12.2)
  ✓ python-dotenv  — installed (1.2.2)
  ✓ ipykernel  — installed (7.2.0)
  ✓ jupyterlab  — installed (4.5.7)

✓ All checks passed.


True

# Chroma Vector Store

Persist embedded chunks in Chroma (embedded/local) and run similarity search. Uses `CHROMA_PERSIST_DIR` from `.env`. Chroma takes an embedding function and handles embedding + storage together.

In [21]:
# Requires: uv add langchain-chroma chromadb
import os
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

persist_dir = str(ROOT / os.getenv("CHROMA_PERSIST_DIR", "./chroma_db"))
store = Chroma(
     collection_name="rag_reference",
     embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
     persist_directory=persist_dir,
 )
# store.add_documents(doc_chunks)                 # doc_chunks from a chunking notebook
results = store.similarity_search("your query", k=3)
for d in results:
    print(d.metadata, d.page_content[:120])

{'trapped': '/False', 'author': 'Prabhukumar Sivamoorthy (Prabhukumarsivamoorthy@gmail.com)', 'page_label': '6', 'keywords': '', 'creator': '(unspecified)', 'page': 5, 'source': '/Users/Prabhukumar/Projects/PycharmProjects/rag-reference/assets/sample-docs/sdlc-end-to-end.pdf', 'creationdate': '2026-05-30T21:30:41-07:00', 'title': 'SDLC End-to-End Reference', 'subject': '(unspecified)', 'producer': 'ReportLab PDF Library - (opensource)', 'total_pages': 12, 'moddate': '2026-05-30T21:30:41-07:00'} SDLC — End-to-End Reference
Page 6

Request/response schemas, status codes and error formats

Rate limiting, idempoten
{'subject': '(unspecified)', 'creator': '(unspecified)', 'author': 'Prabhukumar Sivamoorthy (Prabhukumarsivamoorthy@gmail.com)', 'page': 5, 'page_label': '6', 'source': '/Users/Prabhukumar/Projects/PycharmProjects/rag-reference/assets/sample-docs/sdlc-end-to-end.pdf', 'total_pages': 12, 'title': 'SDLC End-to-End Reference', 'moddate': '2026-05-30T21:30:41-07:00', 'trapped': '/

# Examples(PDF)


### 1.Load the documents

In [22]:
from langchain_community.document_loaders import PyPDFLoader

file_path = ROOT / "assets/sample-docs/sdlc-end-to-end.pdf"

loader = PyPDFLoader(str(file_path))
documents = loader.load()
print(f"Loaded {len(documents)} page(s)")

Loaded 12 page(s)


### 2. Chuncking(Recursive Chunking)

In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators= ["\n\n", "\n", " ", ""]
)

spliting the documents

In [24]:
doc_chunks = text_splitter.split_documents(documents)


### 3.Embeddings

In [25]:
from langchain_openai import OpenAIEmbeddings

openai_embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

### 4. Vector store

In [26]:
import os
from langchain_chroma import Chroma

# Anchor the DB to the repo root so it lands in the same place from any notebook.
persist_dir = str(ROOT / os.getenv("CHROMA_PERSIST_DIR", "./chroma_db"))

vector_store = Chroma(
    collection_name="rag_reference",
    embedding_function=embedding_model,
    persist_directory=persist_dir,
)
print("Collection:", vector_store._collection.name)
print("Persist dir:", persist_dir)

Collection: rag_reference
Persist dir: /Users/Prabhukumar/Projects/PycharmProjects/rag-reference/chroma_db


createing embeddings and storing in vector_store

In [27]:
# Embed the chunks and write them into the persistent collection.
# (Re-running adds duplicates; reset the collection first if you re-ingest.)
ids = vector_store.add_documents(doc_chunks)

print(f"Added {len(ids)} chunks to collection '{vector_store._collection.name}'")
print("Total in collection:", vector_store._collection.count())

Added 29 chunks to collection 'rag_reference'
Total in collection: 58


### Testing

In [34]:
from pprint import pprint

results = store.similarity_search("What is the low level design?", k=3)
for d in results:
    print(d.page_content[:120])

SDLC — End-to-End Reference
Page 5
3. Design
Defines how the system will be built — architecture, components, interfaces
SDLC — End-to-End Reference
Page 5
3. Design
Defines how the system will be built — architecture, components, interfaces
Owner: Senior Developers / Tech Lead  |  Input: HLD  |  Flows to: Implementation

Class/module structure, methods and r
